# Process NP DMS data from Bloom et al.

Import Python modules

In [1]:
import os
import pandas as pd
import numpy as np
from Bio import SeqIO

In [2]:
import yaml

_config_dir = os.path.dirname(os.path.abspath("../config.yaml"))
with open("../config.yaml") as _f:
    _config = yaml.safe_load(_f)
data_dir = os.path.normpath(os.path.join(_config_dir, _config["data_dir"]))

The NP sequence used in the DMS experiment and the one used as the reference for tree building should be the same (Aichi 1968). Read in the DMS data and make sure that this is the case. Then, compute mutational effects.

In [3]:
# Read in the sequence of the reference NP used to build the tree
ref_fasta = os.path.join(data_dir, 'NP/all/curated_reference.fasta')
ref_seq = SeqIO.read(ref_fasta, 'fasta').seq
ref_aa_seq = str(ref_seq.translate())
ref_aa_seq = ref_aa_seq.replace('*', '')
print('Sequence length:', len(ref_aa_seq))

# Read in the DMS data
np_dms_data = pd.read_excel('../data/dms_data/Bloom_NP/Supplementary_file_1.xls')
col_dict = {
    '#SITE' : 'site', 'WT_AA' : 'wt_aa'}
aa_cols = []
for col in np_dms_data.columns:
    if 'PI' in col:
        aa = col[-1]
        col_dict[col] = aa
        aa_cols.append(aa)
np_dms_data.rename(columns=col_dict, inplace=True)

# Correct apparent error in data
np_dms_data['wt_aa'] = np_dms_data['wt_aa'].apply(lambda x: x.replace('N,N,H,H', 'N'))

# Get the sequence of the unmutated protein and make sure that it matches the reference
aa_seq = ''.join(list(np_dms_data['wt_aa']))
assert aa_seq == ref_aa_seq
print('Sequence from DMS data matches the reference sequence used to build the tree')

# Melt dataframe
np_dms_data = pd.melt(
    np_dms_data, id_vars=['site', 'wt_aa'], value_vars=aa_cols,
    var_name='mut_aa', value_name='preference'
)

# Make a smaller dataframe that only has data for rows where the wt_aa == mut_aa
wt_data = np_dms_data[np_dms_data['wt_aa'] == np_dms_data['mut_aa']].copy()
wt_data.rename(columns={'preference' : 'wt_preference'}, inplace=True)

# Merge the wt preferences into the dataframe with all preferences
np_dms_data = np_dms_data.merge(wt_data[['site', 'wt_preference']], on='site')

# Add a column that quantifies the mutation's effect (=preference/wt_preference)
np_dms_data['dms_effect'] = np.log(np_dms_data['preference'] / np_dms_data['wt_preference'])

# Save the data to an output file
output_dir = '../results/dms_data/Bloom_NP/'
if not os.path.isdir(output_dir):
    os.makedirs(output_dir)
output_file = os.path.join(output_dir, 'processed_dms_data.csv')
if not os.path.isfile(output_file):
    np_dms_data.to_csv(output_file, index=False)

np_dms_data.head()

Sequence length: 498
Sequence from DMS data matches the reference sequence used to build the tree


,site,wt_aa,mut_aa,preference,wt_preference,dms_effect
0,1,M,A,0.026720,0.391903,-2.685591
1,2,A,A,0.753436,0.753436,0.000000
2,3,S,A,0.059008,0.116818,-0.682944
3,4,Q,A,0.020215,0.029481,-0.377335
4,5,G,A,0.114320,0.213952,-0.626750
